# Modèle MANOVA / Partie Veljko

Import des différents package

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [3]:
# =============================================================================
# ÉTAPE 1 : PRÉPARATION ET IMPORTATION DES DONNÉES STATIONNARISÉES
# =============================================================================

# 1. Définition du chemin d'accès au fichier (le 'r' devant sert à ignorer les antislashs Windows)
path_parquet_file = "/Users/timeogrienti/economic_research/DATALAKE/PARQUET_FOLDER/main_detrended.parquet"

# 2. Chargement du fichier Parquet dans un DataFrame Pandas
# Le format Parquet est privilégié pour sa rapidité et la conservation des types de données
df = pd.read_parquet(path_parquet_file)

# 3. Sélection des variables pour l'analyse économétrique
# Note : Nous privilégions les versions stationnarisées (suffixe _stat) pour éviter 
# le risque de régression spurieuse (fictive) dû à des tendances non-stationnaires.
df_detrended = df[[
    'year',                  # Variable temporelle pour le contrôle des chocs de période
    'gdp_nominal_stat',      # Variable dépendante (Y) : Croissance du PIB nominal
    'expected_inflation',    # Contrôle : Anticipations des agents (clé pour la courbe de Phillips)
    'taux_changes_stat',     # Contrôle : Canal de transmission externe (Compétitivité-prix)
    'cpi_stat',              # Variable dépendante alternative : Stabilité des prix
    'taux_directeur',        # Variable d'intérêt (X) : Instrument de politique monétaire
    'export_stat',           # Contrôle : Demande extérieure nette
    'import_stat_stat',      # Contrôle : Absorption de la demande domestique
    'yield_perpetual_stat',  # Contrôle : Taux longs (Anticipations de marché et coût du capital)
    'oil_price_stat',        # Contrôle : Chocs d'offre exogènes (Poussée inflationniste)
    'gdp_cycle',             # Indicateur conjoncturel (Écarts à court terme)
    'gdp_trend_stat_stat',   # Contrôle de la croissance potentielle de long terme
    'output_gap'             # Mesure de la pression sur les capacités de production
]].copy()

In [4]:
# 1. Régime "Gold Standard" : on marque d'un '1' toutes les années jusqu'à 1932 inclus
df_detrended['d_gold'] = np.where(df_detrended['year'] <= 1932, 1, 0)

# 2. Régime "Bretton Woods" : on marque d'un '1' l'intervalle 1944-1971
# Note l'usage du '&' pour combiner les deux bornes temporelles
df_detrended['d_woods'] = np.where((df_detrended['year'] >= 1944) & (df_detrended['year'] <= 1971), 1, 0)

# 3. Régime "Floating" (Changes flottants) : on marque d'un '1' tout ce qui suit 1971
df_detrended['d_floating'] = np.where(df_detrended['year'] >= 1972, 1, 0)

**<span style="color:yellow"> On realise des pearson test afin de savoir si les residus de chacune de nos regressions sont corrêlés (explication du pourquoi faire cela dans l'analyse ci dessous**

In [7]:
# Au sein de la partie précédente on suppose que les résidus entre les différentes périodes ne sont pas corrêlés ce qui est empiriquement fortement peu probable.
# On prouve ci dessous que les résidus des 3 régressions sont corrêlés. 

from scipy.stats import pearsonr
e_gdp   = model_gdp_1.resid
e_exch  = model_exch_1.resid
e_inf   = model_inf_1.resid

# Listes des corr avec pearson
corr_gdp_exch, pval_gdp_exch = pearsonr(e_gdp, e_exch)
corr_gdp_inf,  pval_gdp_inf  = pearsonr(e_gdp, e_inf)
corr_exch_inf, pval_exch_inf = pearsonr(e_exch, e_inf)

print("GDP et ex rate : corr =", corr_gdp_exch, ", p =", pval_gdp_exch)
print("GDP et pit : corr =", corr_gdp_inf, ", p =", pval_gdp_inf)
print("ex rate et pit : corr =", corr_exch_inf, ", p =", pval_exch_inf)

# ON peut analyser ci dessous les resultats 

GDP et ex rate : corr = 0.048880873296936474 , p = 0.6204728529258521
GDP et pit : corr = 0.7125490807409403 , p = 1.535944841599208e-17
ex rate et pit : corr = -0.19480878114554825 , p = 0.04643438297801346


#### <span style="color:yellow"> **On analyse ci dessous les résultats du pearson test de la corrélation des résidus des trois regressions** 
Sous l'hypothèse nulle : 

$$
H_0 : \operatorname{corr}(\varepsilon_a, \varepsilon_b) = 0, \quad H_1 : \operatorname{corr}(\varepsilon_a, \varepsilon_b) \neq 0, \quad
\text{avec } a,b \in \{ Y_{\text{GDP}}, Y_{\text{ExchangeRate}}, Y_{\text{Inflation}} \}
$$

Les résidus de la régression sur l'inflation et sur le PIB sont significativement fortement corrêlés. Ainsi il serait plus rigoureux de réaliser une analyse MANOVA (multivariate analysis of variance) pour éviter d'éstimer à tord au sein des régression la redondance des résidus au sein des régressions. Admettons par exemple que les résidus entre deux régressions soient fortement corrêlés comme dans notre cas, il est fortement probable que lorsque l'on éstime l'effet des taux sur chacune des deux variables nous estimions à deux reprise la part d'erreur corrêlés entre les deux modèles, tandis que nous arriverions à omettre cet effet en fusionnant les deux régressions en une. Gràce à la MANOVA nous pourrons donc ne pas sur ou sous estimer les erreurs standards au sein des test de significativités. 

#### **MANOVA CI DESSOUS :** 



### <span style="color:yellow"> **Explication du principe de la MANOVA et ces composants**

Une Manova ou multivariate variance analysis est une anova mais réalisée sur plusieurs variables dépendantes en même temps. Au lieu de tester si les groupes diffèrent sur une seule mesure, on test s'ils diffèrent sur une combinaison de plusieurs mesures en tenant compte des corélations entre elles. 

Ici les groupes sont les différents groupes, et on veut savoir si l'effet des taux sur l'économie (PIB,inflation) diffère entre les différents groupes. 

Ses composantes sont : 

- Wilks' lambda : mesure la proportion de variance non éxpliqué par les groupes. Quel part de comportement du PIB et de l'inflation est non éxpliqué par les régimes ? 
         
- Pillai's trace : Combien de variance de l'inflation et du PIB est éxpliqué à travers le régime X ? (inverse de la question précédente)
 
- Hotelling-Lawley trace : A quelle point les fluctuations entre les régimes dépassents les fluctuations normales ? 
    
- Roy's greatest root : Dans la direction ou les régimes diffèrent le plus ya t-il un effet fort ?

En gros le plus important est la "Pillai's trace" car elle nous permet de mesurer l'intensité des pouvoir prédictif sur la variance de l'économie des différents régimes : 

gold standard : 3,59% et non significatif
Bretton wood : 14,73% significatif
floating rate : 28,83% très significatif


In [8]:
from statsmodels.multivariate.manova import MANOVA

df_manova = pd.concat([df_detrended[['gdp_nominal_stat', 'cpi_stat', 'taux_changes_stat']], X], axis=1)
manova = MANOVA.from_formula('gdp_nominal_stat + cpi_stat ~ gold_int + woods_int + floating_int', data=df_manova)

print(manova.mv_test())

                  Multivariate linear model
                                                             
-------------------------------------------------------------
       Intercept        Value  Num DF  Den DF  F Value Pr > F
-------------------------------------------------------------
          Wilks' lambda 0.7334 2.0000 100.0000 18.1774 0.0000
         Pillai's trace 0.2666 2.0000 100.0000 18.1774 0.0000
 Hotelling-Lawley trace 0.3635 2.0000 100.0000 18.1774 0.0000
    Roy's greatest root 0.3635 2.0000 100.0000 18.1774 0.0000
-------------------------------------------------------------
                                                             
-------------------------------------------------------------
        gold_int        Value  Num DF  Den DF  F Value Pr > F
-------------------------------------------------------------
          Wilks' lambda 0.9631 2.0000 100.0000  1.9179 0.1523
         Pillai's trace 0.0369 2.0000 100.0000  1.9179 0.1523
 Hotelling-Lawley trace 0.

#### <span style="color:yellow">**ANALYSE RESULTAT MANOVA**
Ces resultats sont plutot concluants : 
gold_int :  pas d’effet significatif des taux sur les variables d'interets durant la période gold standard.
woods_int : effet significatif à 1,2% des taux d'interet sur les variables d'interet durant la periode bretton wood.
floating_int : effet extremement significatif à mois de 0.000% des taux d'interet sur les variables d'interet durant la periode floating rate. 

Ceci prend sens sur le plan macroéconomique étant donnée qu'il permettent de mettre en avant que l'effet des taux augmente lorsque la politique monétaire s'éloigne de la gestion de la politique monétaire à travers l'or. 

la MANOVA nous permet de valider un postulat super important pour notre étude étant qu'au dela de l'effet plus ou moins bon des différentes politiques monétaires sur la minimisation du cout social, la politique monétaire entraine une variation dans l'effet de transmissiond des taux sur les agrégats économique. 

**<span style="color:yellow">CONCLUSION D'ANALYSE MANOVA : LE REGIME À DONC BIEN UN REEL IMPACT SUR LA MANIERE DONT LES TAUX IMPACTENT LES AGREGATS ÉTUDIÉS**